In [1]:
import logging
import os
from contextlib import contextmanager

import duckdb

# Configure structured logging (force=True so re-running this cell in an
# already-running kernel doesn't just add duplicate handlers).
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger("EcoLens.DataPipeline")

DEFAULT_DB_PATH = (
    "/Users/macbook/Project/personal/EcoLens/services/data-pipeline"
    "/data/historical/ecolens_historical.duckdb"
)
db_path = os.getenv("DUCKDB_PATH", DEFAULT_DB_PATH)


@contextmanager
def duckdb_connection(path: str, read_only: bool = True):
    """Connect, log, and guarantee the close+log happens exactly once --
    defined here so every later cell does `with duckdb_connection(...) as con:`
    instead of re-pasting this same try/except/finally block (which is how
    the notebook ended up with two copies of the same ~40 lines, and the
    same "DuckDB connection closed cleanly." log line, drifting out of
    sync with each other after only one of them got edited).
    """
    resolved_path = os.path.expanduser(path)
    if read_only and not os.path.exists(resolved_path):
        logger.error("DuckDB database file not found at path: %s", resolved_path)
        raise FileNotFoundError(f"Database file not found: {resolved_path}")

    con = None
    try:
        con = duckdb.connect(database=resolved_path, read_only=read_only)
        logger.info(
            "Successfully connected (read_only=%s) to %s", read_only, resolved_path
        )
        yield con
    except duckdb.Error as e:
        logger.exception("DuckDB internal error while connecting to %s", resolved_path)
        raise ConnectionError(f"Failed to connect to DuckDB database: {e}") from e
    finally:
        if con is not None:
            con.close()
            logger.info("DuckDB connection closed cleanly.")


with duckdb_connection(db_path) as con:
    version = con.execute("SELECT version()").fetchone()[0]
    logger.info("DuckDB Version active: %s", version)

    tables_df = con.execute(
        "SELECT table_name FROM duckdb_tables WHERE internal = false"
    ).df()
    print("Tables in database:")
    for name in tables_df["table_name"]:
        print(f" - {name}")


2026-07-25 20:26:45,743 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-25 20:26:45,744 [INFO] EcoLens.DataPipeline: DuckDB Version active: v1.5.5
2026-07-25 20:26:46,298 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


Tables in database:
 - aemo_holidays
 - aemo_nem_dispatch
 - aemo_wem_dispatch
 - bom_observations
 - openelectricity_responses


In [4]:
# Bug fixed: the previous version interpolated START/END *unquoted*
# into the SQL string (f"... ts >= {START} ..."), which DuckDB parses as
# a bare arithmetic/date expression, not a string literal -- it either
# raises `ParserException: syntax error at or near "00"` on a clean
# kernel, or silently returns wrong rows if some other stale state
# happens to paper over it. Use `?` placeholders instead: DuckDB
# binds them as actual parameters, so quoting is never the caller's
# problem.
START = "2026-06-01 00:00:00"
END = "2026-06-30 00:00:00"

query = """
    SELECT
        CAST(ts AS DATE) AS data_date,
        COUNT(*) AS record_frequency
    FROM aemo_nem_dispatch
    WHERE ts >= ?
      AND ts <  ?
    GROUP BY CAST(ts AS DATE)
    ORDER BY data_date ASC
"""
query_params = [START, END]


In [5]:
with duckdb_connection(db_path) as con:
    df = con.execute(query, query_params).df()

df


2026-07-25 20:29:01,789 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-25 20:29:01,871 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


,data_date,record_frequency
0,2026-06-01,1728
1,2026-06-02,1728
2,2026-06-03,1728
3,2026-06-04,1728
4,2026-06-05,1728
5,2026-06-06,1728
6,2026-06-07,1728
7,2026-06-08,1728
8,2026-06-09,726
9,2026-06-10,1002
